# Day 48 — Decision trees, random forests and gradient boosting

A decision tree is the only model in this series with no coefficients and no distance
function. It asks questions — *is column 7 below 0.31?* — and keeps asking until it is
willing to commit to an answer. That makes it the one model you can read, and the one
model whose every self-report is wrong in a direction you can predict.

The same building block gets used three ways:

```
one tree          : ask questions until every leaf is pure   -> it memorises
many independent  : average trees that memorise different noise  (random forest)
many dependent    : each small tree fixes what the last ones missed  (boosting)
```

The forest exists to kill variance, boosting exists to kill bias, and swapping their
tree depths swaps their failure modes. Along the way: a training score that an
unrestricted tree drives to 1.000 unless the data actively stops it, a feature-importance table that
ranks a pure-noise column first, a model that cannot predict a number it has never
seen, and a loss curve that falls forever while the model gets worse.

## Impurity, and the one split

Gini and entropy both ask "how mixed is this node" and both hit 0 when it is pure. The
split that wins is the one buying the biggest weighted drop. `variance` is the same
idea for regression: minimising weighted variance is minimising squared error, because
the best constant for a leaf is its mean (Day 45).

In [1]:
# ================================================================ 1. impurity
def gini(y, k):
    """1 - sum p^2.  Zero when a node holds one class, 1 - 1/k when it is uniform."""
    if len(y) == 0:
        return 0.0
    p = np.bincount(y, minlength=k) / len(y)
    return float(1.0 - np.sum(p * p))


def entropy(y, k):
    if len(y) == 0:
        return 0.0
    p = np.bincount(y, minlength=k) / len(y)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)) + 0.0)   # keeps a pure node at 0.0, not -0.0


def variance(y, k=None):
    """The regression criterion.  Minimising weighted variance is the same thing as
    minimising squared error, because the best constant prediction for a leaf is its
    mean (Day 45 again)."""
    return 0.0 if len(y) == 0 else float(np.var(y))


CRITERIA = {'gini': gini, 'entropy': entropy, 'variance': variance}


# ================================================================ 2. one split
def split_candidates(col):
    """Midpoints between consecutive distinct values.  A column with m distinct
    values offers m - 1 thresholds - which is the whole of section 7."""
    u = np.unique(col)
    return (u[:-1] + u[1:]) / 2.0 if len(u) > 1 else np.empty(0)


import math
import numpy as np
np.set_printoptions(precision=4, suppress=True)

for name in ('gini', 'entropy'):
    vals = [CRITERIA[name](np.array([0] * i + [1] * (8 - i)), 2) for i in range(5)]
    print(f'{name:>8} for 0/8, 1/7, 2/6, 3/5, 4/4 of 8 rows:', [round(v, 3) for v in vals])
print()
print('a column of 5 distinct values offers 4 thresholds:',
      np.round(split_candidates(np.array([1.0, 2.0, 2.0, 5.0, 9.0, 11.0])), 2))

    gini for 0/8, 1/7, 2/6, 3/5, 4/4 of 8 rows: [0.0, 0.219, 0.375, 0.469, 0.5]
 entropy for 0/8, 1/7, 2/6, 3/5, 4/4 of 8 rows: [0.0, 0.544, 0.811, 0.954, 1.0]

a column of 5 distinct values offers 4 thresholds: [ 1.5  3.5  7.  10. ]


## Finding the best split twice

The obvious implementation rebuilds both children for every candidate threshold: O(n)
work times O(n) candidates times d columns, so **O(n² d) per node**. Sorting the column
once and sweeping the cut from left to right makes every statistic the criterion needs a
running total, so each candidate is O(1) and the node costs **O(n log n · d)**.

For gini and entropy the running totals are class counts; for variance they are `sum(y)`
and `sum(y²)`, because `var = E[y²] − E[y]²`. Both versions are below and they agree
exactly — only the fast one is usable at the sizes further down.

In [2]:
def best_split_naive(X, y, k, crit, feats=None, min_leaf=1):
    """Exhaustive search, written the obvious way: for every candidate threshold,
    rebuild both children and score them.  O(n) work per candidate and O(n)
    candidates per feature, so O(n^2 d) per node.  Section 3 replaces it."""
    fn = CRITERIA[crit]
    n = len(y)
    parent = fn(y, k)
    best = None
    for j in (range(X.shape[1]) if feats is None else feats):
        col = X[:, j]
        for t in split_candidates(col):
            left = col <= t
            nl = int(left.sum())
            if nl < min_leaf or n - nl < min_leaf:
                continue
            child = (nl * fn(y[left], k) + (n - nl) * fn(y[~left], k)) / n
            gain = parent - child
            if best is None or gain > best[0] + 1e-12:
                best = (gain, j, float(t))
    return best


def best_split(X, y, k, crit, feats=None, min_leaf=1):
    """The same answer in O(n log n * d): sort the column once, then sweep the cut
    from left to right.  Every statistic the criterion needs is a running total, so
    moving the cut by one row is O(1) instead of O(n).

    For gini and entropy the running totals are class counts; for variance they are
    sum(y) and sum(y^2), because var = E[y^2] - E[y]^2.
    """
    n = len(y)
    parent = CRITERIA[crit](y, k)
    best = None
    cols = range(X.shape[1]) if feats is None else feats
    for j in cols:
        col = X[:, j]
        order = np.argsort(col, kind='stable')
        cv, yv = col[order], y[order]
        # a cut after position i is only legal where the value actually changes
        cut = np.flatnonzero(cv[:-1] < cv[1:])
        if len(cut) == 0:
            continue
        nl = cut + 1
        ok = (nl >= min_leaf) & (n - nl >= min_leaf)
        if not ok.any():
            continue
        cut, nl = cut[ok], nl[ok]
        if crit == 'variance':
            c1 = np.cumsum(yv)[cut]
            c2 = np.cumsum(yv * yv)[cut]
            t1, t2 = yv.sum(), (yv * yv).sum()
            vl = c2 / nl - (c1 / nl) ** 2
            vr = (t2 - c2) / (n - nl) - ((t1 - c1) / (n - nl)) ** 2
            child = (nl * vl + (n - nl) * vr) / n
        else:
            onehot = np.zeros((n, k))
            onehot[np.arange(n), yv] = 1.0
            cum = np.cumsum(onehot, axis=0)[cut]          # class counts on the left
            tot = onehot.sum(axis=0)
            pl = cum / nl[:, None]
            pr = (tot - cum) / (n - nl)[:, None]
            if crit == 'gini':
                il = 1.0 - np.sum(pl * pl, axis=1)
                ir = 1.0 - np.sum(pr * pr, axis=1)
            else:
                with np.errstate(divide='ignore', invalid='ignore'):
                    il = -np.sum(np.where(pl > 0, pl * np.log2(pl), 0.0), axis=1)
                    ir = -np.sum(np.where(pr > 0, pr * np.log2(pr), 0.0), axis=1)
            child = (nl * il + (n - nl) * ir) / n
        gains = parent - child
        i = int(np.argmax(gains))
        g = float(gains[i])
        if best is None or g > best[0] + 1e-12:
            best = (g, j, float((cv[cut[i]] + cv[cut[i] + 1]) / 2.0))
    return best


import time
rng = np.random.default_rng(0)
bad = 0
for _ in range(60):
    n = int(rng.integers(6, 60)); d = int(rng.integers(1, 4)); k = int(rng.integers(2, 4))
    Xt = np.round(rng.uniform(0, 3, (n, d)), 2); yt = rng.integers(0, k, n)
    for crit in ('gini', 'entropy'):
        a, b = best_split_naive(Xt, yt, k, crit), best_split(Xt, yt, k, crit)
        if (a is None) != (b is None) or (a and abs(a[0] - b[0]) > 1e-9):
            bad += 1
    yv = rng.normal(size=n)
    a, b = best_split_naive(Xt, yv, None, 'variance'), best_split(Xt, yv, None, 'variance')
    if (a is None) != (b is None) or (a and abs(a[0] - b[0]) > 1e-9):
        bad += 1
print('disagreements over 60 problems x {gini, entropy, variance}:', bad)

Xb = rng.uniform(-2, 2, (600, 10)); yb = (1.6 * Xb[:, 0] + 0.5 * Xb[:, 1] > 0).astype(int)
t0 = time.perf_counter(); best_split_naive(Xb, yb, 2, 'gini'); t_slow = time.perf_counter() - t0
t0 = time.perf_counter(); best_split(Xb, yb, 2, 'gini'); t_fast = time.perf_counter() - t0
print(f'one root split on 600x10: naive {1000 * t_slow:.0f} ms, swept {1000 * t_fast:.1f} ms '
      f'({t_slow / t_fast:.0f}x - one measurement)')

disagreements over 60 problems x {gini, entropy, variance}: 0
one root split on 600x10: naive 99 ms, swept 1.5 ms (67x - one measurement)


## The tree itself

Internal nodes hold a question, leaves hold a constant: a majority class, or a mean.
Note what `path` returns — the sequence of questions one prediction actually asked. That
is the whole explanation for one answer, and it is LeetCode 257 with real data in it.

In [3]:
class Node:
    __slots__ = ('feat', 'thr', 'left', 'right', 'value', 'n', 'imp')

    def __init__(self, value, n, imp):
        self.feat = None
        self.thr = None
        self.left = None
        self.right = None
        self.value = value
        self.n = n
        self.imp = imp

    @property
    def is_leaf(self):
        return self.feat is None


class DecisionTree:
    """One tree, for classification (majority vote) or regression (mean)."""

    def __init__(self, task='clf', crit=None, max_depth=None, min_leaf=1,
                 min_gain=0.0, max_features=None, rng=None, n_classes=None):
        self.task = task
        self.crit = crit or ('gini' if task == 'clf' else 'variance')
        self.max_depth = math.inf if max_depth is None else max_depth
        self.min_leaf = min_leaf
        self.min_gain = min_gain
        self.max_features = max_features
        self.rng = rng or np.random.default_rng(0)
        self.k = n_classes
        self.root = None
        self.n_nodes = 0
        self.n_leaves = 0
        self.depth = 0

    def _value(self, y):
        if self.task == 'clf':
            return int(np.bincount(y, minlength=self.k).argmax())
        return float(y.mean())

    def fit(self, X, y):
        if self.task == 'clf' and self.k is None:
            self.k = int(y.max()) + 1
        self.n_nodes = self.n_leaves = self.depth = 0
        self.root = self._grow(X, y, 0)
        return self

    def _grow(self, X, y, d):
        node = Node(self._value(y), len(y), CRITERIA[self.crit](y, self.k))
        self.n_nodes += 1
        self.depth = max(self.depth, d)
        pure = node.imp <= 1e-12
        if pure or d >= self.max_depth or len(y) < 2 * self.min_leaf:
            self.n_leaves += 1
            return node
        feats = None
        if self.max_features is not None and self.max_features < X.shape[1]:
            feats = self.rng.choice(X.shape[1], self.max_features, replace=False)
        sp = best_split(X, y, self.k, self.crit, feats, self.min_leaf)
        if sp is None or sp[0] <= self.min_gain:
            self.n_leaves += 1
            return node
        _, j, t = sp
        node.feat, node.thr = j, t
        mask = X[:, j] <= t
        node.left = self._grow(X[mask], y[mask], d + 1)
        node.right = self._grow(X[~mask], y[~mask], d + 1)
        return node

    def predict(self, X):
        out = np.empty(len(X), dtype=int if self.task == 'clf' else float)
        for i, row in enumerate(X):
            node = self.root
            while not node.is_leaf:
                node = node.left if row[node.feat] <= node.thr else node.right
            out[i] = node.value
        return out

    def path(self, row, names=None):
        """The root-to-leaf path a single prediction takes - the tree's whole
        explanation for one answer.  This is LeetCode 257 with real data in it."""
        node, steps = self.root, []
        while not node.is_leaf:
            nm = names[node.feat] if names else f'x{node.feat}'
            go_left = row[node.feat] <= node.thr
            steps.append(f'{nm} {"<=" if go_left else ">"} {node.thr:.3f}')
            node = node.left if go_left else node.right
        return steps, node.value

    def importances(self, n_features):
        """Impurity-based importance: how much weighted impurity each feature's
        splits removed, normalised.  Section 7 is about how this lies."""
        imp = np.zeros(n_features)

        def walk(node):
            if node.is_leaf:
                return
            drop = node.n * node.imp - node.left.n * node.left.imp - node.right.n * node.right.imp
            imp[node.feat] += drop
            walk(node.left)
            walk(node.right)

        walk(self.root)
        s = imp.sum()
        return imp / s if s > 0 else imp


# ================================================================ 5. data sets
def stripes(n=400, seed=0, noise=0.0):
    """Two classes in diagonal bands: a tree has to approximate the diagonal with a
    staircase, which is the honest picture of what axis-aligned splits can express."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3, 3, (n, 2))
    y = ((X[:, 0] + X[:, 1]) > 0).astype(int)
    if noise:
        flip = rng.random(n) < noise
        y[flip] = 1 - y[flip]
    return X, y


def accuracy(a, b):
    return float(np.mean(a == b))


X, y = stripes(400, seed=0, noise=0.10)
t3 = DecisionTree(max_depth=3).fit(X, y)
print(f'depth-3 tree on 400 points: {t3.n_nodes} nodes, {t3.n_leaves} leaves')
steps, val = t3.path(X[0], names=['x', 'y'])
print(f'the prediction for {np.round(X[0], 3).tolist()} is one root-to-leaf path:')
for s in steps:
    print('   ', s)
print('    -> class', val)
sp = best_split(X, y, 2, 'gini')
print(f'\nbest split overall: x{sp[1]} <= {sp[2]:.4f}, gini gain {sp[0]:.4f}')

depth-3 tree on 400 points: 15 nodes, 8 leaves
the prediction for [0.822, -1.381] is one root-to-leaf path:
    x > 0.255
    y > -1.663
    x <= 1.252
    -> class 1

best split overall: x0 <= 0.2551, gini gain 0.0757


## Silent failure #1: a tree grown to purity scores 1.000 on its own data

Training accuracy is monotone in depth and ends at exactly 1.000, and that number
carries no information whatsoever. It gets there whenever every impure node still offers
a split with positive gain. Two things can stop that: exact symmetry — the cell below
fits a perfectly symmetric XOR lattice where every candidate split is worth exactly
nothing and the tree stops at a single leaf — and, far more commonly, two identical rows
that disagree about the label, which leave an impure node with no legal split at all.

This is Day 47's k = 1 nearest neighbour wearing a different hat: a fully grown tree is
a lookup table for its own training set.

In [4]:
perm = np.random.default_rng(0).permutation(400)
tr, te = perm[:260], perm[260:]
print(f'{"max_depth":>10} {"leaves":>7} {"train":>7} {"test":>7}')
for d in (1, 2, 3, 5, 8, 12, None):
    t = DecisionTree(max_depth=d).fit(X[tr], y[tr])
    print(f'{str(d):>10} {t.n_leaves:>7} {accuracy(t.predict(X[tr]), y[tr]):>7.3f} '
          f'{accuracy(t.predict(X[te]), y[te]):>7.3f}')

 max_depth  leaves   train    test
         1       2   0.704   0.664
         2       4   0.781   0.814
         3       8   0.858   0.829
         5      24   0.912   0.850
         8      42   0.973   0.771
        12      50   0.996   0.764
      None      52   1.000   0.764


In [5]:
def small24():
    """24 hand-placed points, the set the interactive page grows a tree on.  Class 1
    fills the lower-right block except for one row at (0.70, 0.30) labelled 0 - which
    is the whole of the "memorise one row" story."""
    P = [(0.10, 0.15, 0), (0.18, 0.45, 0), (0.12, 0.72, 0), (0.26, 0.88, 0),
         (0.22, 0.28, 0), (0.08, 0.58, 0), (0.33, 0.18, 0), (0.36, 0.66, 0),
         (0.30, 0.40, 0),
         (0.62, 0.12, 1), (0.74, 0.22, 1), (0.88, 0.16, 1), (0.66, 0.34, 1),
         (0.82, 0.40, 1), (0.94, 0.28, 1), (0.58, 0.44, 1),
         (0.60, 0.66, 0), (0.72, 0.78, 0), (0.86, 0.70, 0), (0.92, 0.88, 0),
         (0.64, 0.92, 0), (0.78, 0.58, 0), (0.90, 0.56, 0),
         (0.70, 0.30, 0)]
    return np.array([[r[0], r[1]] for r in P]), np.array([r[2] for r in P])


def grow_trace(X, y, max_depth=6):
    """Grow a tree depth-first and report one row per split, in the order it is made:
    (depth, feature, threshold, gini drop, leaves afterwards, training accuracy)."""
    k = int(y.max()) + 1
    rows = []
    root = {'X': X, 'y': y}
    leaves = [root]

    def acc():
        ok = sum(int((lf['y'] == np.bincount(lf['y'], minlength=k).argmax()).sum())
                 for lf in leaves)
        return ok / len(y)

    def rec(node, d):
        Xn, yn = node['X'], node['y']
        if d >= max_depth or gini(yn, k) <= 1e-12:
            return
        sp = best_split(Xn, yn, k, 'gini')
        if sp is None or sp[0] <= 1e-12:
            return
        gain, j, t = sp
        m = Xn[:, j] <= t
        left = {'X': Xn[m], 'y': yn[m]}
        right = {'X': Xn[~m], 'y': yn[~m]}
        i = next(q for q, lf in enumerate(leaves) if lf is node)
        leaves[i:i + 1] = [left, right]
        rows.append((d, j, float(t), gain, len(leaves), acc()))
        rec(left, d + 1)
        rec(right, d + 1)

    rec(root, 0)
    return rows


def xor_lattice():
    """A perfectly symmetric 8x8 XOR lattice.  Every candidate split is worth exactly
    zero here, which is the one way a tree can fail to reach 1.000 on clean data."""
    pts = [(-0.875 + i * 0.25, -0.875 + j * 0.25) for i in range(8) for j in range(8)]
    X = np.array(pts)
    return X, np.array([int((p[0] > 0) != (p[1] > 0)) for p in pts])


X24, y24 = small24()
base24 = float(np.mean(y24 == np.bincount(y24).argmax()))
print(f'24 points; calling everything the majority class scores {base24:.4f}')
print(f'{"cut":>4} {"depth":>6} {"split":>18} {"gini drop":>11} {"leaves":>7} {"accuracy":>9}')
for i, (d, feat, thr, gain, nleaf, acc) in enumerate(grow_trace(X24, y24)):
    nm = 'x' if feat == 0 else 'y'
    print(f'{i + 1:>4} {d:>6} {nm + " <= " + format(thr, ".3f"):>18} {gain:>11.5f} '
          f'{nleaf:>7} {acc:>9.4f}')
print()
print('and one of the two ways a tree can stop short of 1.000 - a perfectly symmetric')
print('lattice, the other being identical rows that disagree about the label:')
XL, yL = xor_lattice()
tL = DecisionTree().fit(XL, yL)
print(f'  best available gain {best_split(XL, yL, 2, "gini")[0]:.1f}, '
      f'{tL.n_leaves} leaf, train accuracy {accuracy(tL.predict(XL), yL):.4f}')

24 points; calling everything the majority class scores 0.7083
 cut  depth              split   gini drop  leaves  accuracy
   1      0         y <= 0.445     0.17014       2    0.7917
   2      1         x <= 0.455     0.34028       3    0.9583
   3      2         x <= 0.720     0.03125       4    0.9583
   4      3         x <= 0.680     0.37500       5    1.0000

and one of the two ways a tree can stop short of 1.000 - a perfectly symmetric
lattice, the other being identical rows that disagree about the label:
  best available gain 0.0, 1 leaf, train accuracy 0.5000


## Silent failure #2: greedy cannot see one step ahead

`y = (x0 > 0) XOR (x1 > 0)`, with no label noise at all. A depth-2 tree represents this
exactly: split x0 at 0, then x1 at 0 in both children, four pure leaves. Greedy
top-down induction never finds it, because at the root **neither column carries any
information on its own** — so the first split is chosen on sampling noise, and it lands
nowhere near 0.

Representable and unreachable at the same time. Finding the optimal tree is NP-hard,
which is why essentially every tree learner in production is greedy. Exact solvers do
exist — MIP formulations, DL8.5, MurTree, GOSDT — but they are a different tool for a
different size of problem.

In [6]:
def xor_data(n=400, seed=1, noise=0.0):
    """y = sign(x0) XOR sign(x1).  Neither feature carries any information on its own,
    which is exactly the case a greedy top-down search cannot see."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, (n, 2))
    y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(int)
    if noise:
        flip = rng.random(n) < noise
        y[flip] = 1 - y[flip]
    return X, y


Xx, yx = xor_data(400, seed=1)
par = gini(yx, 2)
for j in (0, 1):
    m = Xx[:, j] <= 0.0
    g = par - (m.sum() * gini(yx[m], 2) + (~m).sum() * gini(yx[~m], 2)) / len(yx)
    print(f'gain of the correct split on x{j} at 0:  {g:.8f}')
g, j, thr = best_split(Xx, yx, 2, 'gini')
print(f'gain of the split greedy picks:          {g:.8f}   (x{j} <= {thr:.4f})')
print()
oracle = ((Xx[:, 0] > 0) ^ (Xx[:, 1] > 0)).astype(int)
print('the correct depth-2 tree scores', accuracy(oracle, yx))
for d in (2, 3, 4, 6):
    t = DecisionTree(max_depth=d).fit(Xx, yx)
    print(f'greedy depth-{d} scores {accuracy(t.predict(Xx), yx):.4f}  ({t.n_leaves} leaves)')

gain of the correct split on x0 at 0:  0.00000124
gain of the correct split on x1 at 0:  0.00061250
gain of the split greedy picks:          0.01114311   (x0 <= -0.9414)

the correct depth-2 tree scores 1.0
greedy depth-2 scores 0.5675  (3 leaves)
greedy depth-3 scores 0.6575  (5 leaves)
greedy depth-4 scores 0.9400  (7 leaves)
greedy depth-6 scores 1.0000  (10 leaves)


## Silent failure #3: feature importance rewards opportunity, not use

One binary column that *is* the answer (20% of labels flipped), one continuous column of
pure noise, three binary noise columns. Impurity importance ranks the pure-noise column
**first**.

The mechanism is the candidate count. A continuous column with 210 distinct values
offers 209 thresholds at the root; a binary column offers 1. Impurity importance counts
how much each column got to remove — it has no way to ask whether removing it meant
anything. Permutation importance asks exactly that, and it has to be measured on rows
the model never saw, or it will happily reward memorisation too.

In [7]:
def cardinality_data(n=300, seed=2):
    """One binary column that *is* the answer (with a little noise), one continuous
    column of pure noise, and three more binary columns of pure noise.

    Columns: 0 = the real signal (binary), 1 = continuous noise, 2..4 = binary noise.
    """
    rng = np.random.default_rng(seed)
    signal = rng.integers(0, 2, n)
    y = signal.copy()
    flip = rng.random(n) < 0.20
    y[flip] = 1 - y[flip]
    X = np.column_stack([signal.astype(float), rng.random(n),
                         rng.integers(0, 2, (n, 3)).astype(float)])
    return X, y, ['real (binary)', 'noise (continuous)', 'noise b1', 'noise b2', 'noise b3']


def permutation_importance(model, X, y, rng, repeats=10, score=accuracy):
    """Shuffle one column at a time and watch the score fall.  It asks the only
    question that matters - does the model actually use this column - instead of
    counting how many chances the column had to be picked."""
    base = score(model.predict(X), y)
    out = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        drops = []
        for _ in range(repeats):
            Xp = X.copy()
            Xp[:, j] = rng.permutation(Xp[:, j])
            drops.append(base - score(model.predict(Xp), y))
        out[j] = np.mean(drops)
    return out


Xc, yc, names = cardinality_data(300, seed=2)
rc = np.random.default_rng(0).permutation(300)
ctr, cte = rc[:210], rc[210:]
tc = DecisionTree().fit(Xc[ctr], yc[ctr])
imp = tc.importances(Xc.shape[1])
pi = permutation_importance(tc, Xc[cte], yc[cte], np.random.default_rng(7), repeats=20)
print('distinct values per column:', [len(np.unique(Xc[:, j])) for j in range(Xc.shape[1])])
print('thresholds offered at the root:',
      [len(split_candidates(Xc[ctr, j])) for j in range(Xc.shape[1])])
print()
print(f'{"column":<22} {"impurity":>10} {"permutation (held out)":>24}')
for nm, a, b in zip(names, imp, pi):
    print(f'{nm:<22} {a:>10.3f} {b:>+24.3f}')
print()
print(f'train {accuracy(tc.predict(Xc[ctr]), yc[ctr]):.3f}  test '
      f'{accuracy(tc.predict(Xc[cte]), yc[cte]):.3f}')

distinct values per column: [2, 300, 2, 2, 2]
thresholds offered at the root: [1, 209, 1, 1, 1]

column                   impurity   permutation (held out)
real (binary)               0.347                   +0.168
noise (continuous)          0.580                   +0.033
noise b1                    0.022                   +0.007
noise b2                    0.025                   -0.019
noise b3                    0.026                   +0.006

train 1.000  test 0.656


## Silent failure #4: one row changes the whole tree

Refit 200 times, each time with a single row removed. The *root* split changes in a
large fraction of them. Nothing about the data changed meaningfully — 199 of 200 rows
are identical every time.

A model this sensitive to its sample is a model with high variance, and high variance is
exactly what averaging is good at removing. That is the entire argument for the forest.

In [8]:
Xi, yi = stripes(200, seed=0, noise=0.10)
base = DecisionTree(max_depth=4).fit(Xi, yi)
changed, drift = 0, []
for i in range(len(yi)):
    keep = np.ones(len(yi), bool); keep[i] = False
    t = DecisionTree(max_depth=4).fit(Xi[keep], yi[keep])
    if (t.root.feat, round(t.root.thr, 9)) != (base.root.feat, round(base.root.thr, 9)):
        changed += 1
    drift.append(float(np.mean(t.predict(Xi) != base.predict(Xi))))
print(f'base root: x{base.root.feat} <= {base.root.thr:.4f}, {base.n_leaves} leaves')
print(f'root split changed in {changed} of 200 single-row refits ({100 * changed / 200:.0f}%)')
print(f'predictions moved on {100 * np.mean(drift):.1f}% of rows on average, '
      f'{100 * max(drift):.1f}% at worst')

base root: x1 <= -0.3914, 16 leaves
root split changed in 60 of 200 single-row refits (30%)
predictions moved on 1.8% of rows on average, 7.5% at worst


## The forest: bagging first, then making the trees ask different questions

Bagging makes the trees see different rows. `max_features` makes them ask different
questions — and without it, on data with one dominant column, **every tree opens with
the same split**. Averaging predictors that agree buys nothing, so the disagreement
number below is the quantity being paid for.

Out-of-bag scoring is free: each bootstrap sample leaves out about `(1 − 1/n)^n → 1/e =
36.8%` of the rows, so every row is unseen by roughly a third of the trees.

The cell below uses a single forest seed and 30 trees so that it runs quickly; `trees.py`
averages three seeds and pools 90 root choices, so its numbers are close but not
identical to these.

In [9]:
# ================================================================ 3. the forest
class RandomForest:
    """Bagging plus a random subset of features at every split.

    The two knobs do different jobs.  Bagging makes the trees see different rows;
    max_features makes them ask different questions.  Without the second one a few
    dominant features get picked by every tree, the trees agree, and averaging
    agreeing predictors buys nothing.
    """

    def __init__(self, n_trees=30, max_features='sqrt', max_depth=None, min_leaf=1,
                 bootstrap=True, seed=0, task='clf'):
        self.n_trees = n_trees
        self.max_features = max_features
        self.max_depth = max_depth
        self.min_leaf = min_leaf
        self.bootstrap = bootstrap
        self.seed = seed
        self.task = task
        self.trees = []
        self.oob_idx = []

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n, d = X.shape
        k = int(y.max()) + 1 if self.task == 'clf' else None
        mf = (max(1, int(round(math.sqrt(d)))) if self.max_features == 'sqrt'
              else (d if self.max_features is None else self.max_features))
        self.trees, self.oob_idx = [], []
        for _ in range(self.n_trees):
            idx = rng.integers(0, n, n) if self.bootstrap else np.arange(n)
            oob = np.setdiff1d(np.arange(n), idx) if self.bootstrap else np.empty(0, int)
            t = DecisionTree(task=self.task, max_depth=self.max_depth, min_leaf=self.min_leaf,
                             max_features=mf, rng=np.random.default_rng(rng.integers(1 << 30)),
                             n_classes=k).fit(X[idx], y[idx])
            self.trees.append(t)
            self.oob_idx.append(oob)
        self.k = k
        return self

    def predict(self, X):
        P = np.array([t.predict(X) for t in self.trees])
        if self.task == 'reg':
            return P.mean(axis=0)
        return np.array([np.bincount(P[:, i], minlength=self.k).argmax() for i in range(len(X))])

    def oob_score(self, X, y):
        """Every row was left out of about 37% of the trees, so the forest comes with
        a held-out estimate that costs nothing: 1 - (1 - 1/n)^n -> 1/e."""
        votes = np.zeros((len(X), self.k), dtype=int)
        for t, oob in zip(self.trees, self.oob_idx):
            if len(oob):
                for i, p in zip(oob, t.predict(X[oob])):
                    votes[i, p] += 1
        seen = votes.sum(axis=1) > 0
        return float(np.mean(votes[seen].argmax(axis=1) == y[seen])), int(seen.sum())

    def disagreement(self, X):
        """Mean fraction of rows on which two different trees disagree.  This is the
        quantity averaging is paid to exploit: identical trees have nothing to average."""
        P = np.array([t.predict(X) for t in self.trees])
        m = len(P)
        tot = [float(np.mean(P[a] != P[b])) for a in range(m) for b in range(a + 1, m)]
        return float(np.mean(tot))


def dominant(n=900, seed=0, noise=0.08):
    """Ten columns: one that dominates the answer, four that help a little, five pure
    noise.  The shape of most tabular data, and the shape that makes feature
    subsampling matter - without it every tree picks the same root."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, 10))
    score = 3.0 * X[:, 0] + 0.55 * X[:, 1] + 0.5 * X[:, 2] - 0.45 * X[:, 3] + 0.4 * X[:, 4]
    y = (score > 0).astype(int)
    flip = rng.random(n) < noise
    y[flip] = 1 - y[flip]
    return X, y


Xd, yd = dominant(900, seed=0)
rd = np.random.default_rng(1).permutation(900)
dtr, dte = rd[:600], rd[600:]
one = DecisionTree().fit(Xd[dtr], yd[dtr])
print(f'{"model":<32} {"test":>7} {"disagree":>9} {"oob":>7}')
print(f'{"single unpruned tree":<32} {accuracy(one.predict(Xd[dte]), yd[dte]):>7.3f} '
      f'{"-":>9} {"-":>7}')
for nm, kw in [('bagging only, all 10 features', dict(max_features=None)),
               ('random forest, sqrt = 3', dict(max_features='sqrt'))]:
    f = RandomForest(n_trees=30, seed=0, **kw).fit(Xd[dtr], yd[dtr])
    print(f'{nm:<32} {accuracy(f.predict(Xd[dte]), yd[dte]):>7.3f} '
          f'{f.disagreement(Xd[dte]):>9.3f} {f.oob_score(Xd[dtr], yd[dtr])[0]:>7.3f}')
    print('    root column of each tree:',
          np.bincount([t.root.feat for t in f.trees], minlength=10).tolist())

model                               test  disagree     oob
single unpruned tree               0.773         -       -
bagging only, all 10 features      0.853     0.237   0.820
    root column of each tree: [30, 0, 0, 0, 0, 0, 0, 0, 0, 0]
random forest, sqrt = 3            0.880     0.319   0.838
    root column of each tree: [8, 6, 1, 7, 4, 0, 0, 1, 2, 1]


## Silent failure #5: a tree cannot predict a number it has never seen

A leaf holds a constant, so the model is a step function and the last step runs flat
forever. Averaging flat things is flat, so the forest does not fix it. A tree also
cannot go above its largest leaf mean — any model whose output is a non-negative
weighted average of training targets has this property.

The usual fix is not a better tree. It is to model the trend separately and let the tree
fit what is left.

In [10]:
def wave(n=120, lo=-3.0, hi=3.0, seed=3, noise=0.15):
    rng = np.random.default_rng(seed)
    x = np.sort(rng.uniform(lo, hi, n))
    return x[:, None], np.sin(x) + 0.35 * x + noise * rng.standard_normal(n)


Xw, yw = wave(120, seed=3)
tw = DecisionTree(task='reg', max_depth=6).fit(Xw, yw)
fw = RandomForest(n_trees=40, task='reg', max_features=None, seed=0).fit(Xw, yw)
print(f'{"x":>7} {"tree":>9} {"forest":>9} {"truth":>9}')
for xq in (2.9, 3.5, 5.0, 10.0, 100.0):
    print(f'{xq:>7} {float(tw.predict(np.array([[xq]]))[0]):>+9.4f} '
          f'{float(fw.predict(np.array([[xq]]))[0]):>+9.4f} '
          f'{math.sin(xq) + 0.35 * xq:>+9.4f}')
print(f'\nlargest y in the data {yw.max():.4f}, largest prediction {tw.predict(Xw).max():.4f}')

      x      tree    forest     truth
    2.9   +1.3829   +1.3301   +1.2542
    3.5   +1.3829   +1.3301   +0.8742
    5.0   +1.3829   +1.3301   +0.7911
   10.0   +1.3829   +1.3301   +2.9560
  100.0   +1.3829   +1.3301  +34.4936

largest y in the data 1.7795, largest prediction 1.5463


## Boosting: gradient descent where the parameter is a function

Day 45 took steps in parameter space, `w ← w − lr · dL/dw`. Here the model is a sum of
trees and the thing being improved is the function `F`:

```
F0       = mean(y)                       the best constant under squared loss
residual = y − F                         = −dL/dF for L = ½(y − F)²
F        = F + lr · tree(X → residual)   one small step in function space
```

"Fit the next tree to what is left over" is not a separate idea — it is what the
gradient of squared loss happens to look like. Change the loss and the target of the
next tree changes with it. That is the whole content of the word *gradient*.

In [11]:
# ================================================================ 4. boosting
class GradientBoosting:
    """Gradient descent, where the thing being descended is a function.

    Day 45 took a step in parameter space: w -= lr * dL/dw.  Here the model is a sum
    of trees, the "parameter" is the function F itself, and the step is a small tree
    fitted to -dL/dF.  For squared loss that derivative is exactly the residual, which
    is why the textbook version looks like "fit the next tree to what is left over".
    """

    def __init__(self, n_trees=60, lr=0.1, max_depth=2, min_leaf=1, seed=0):
        self.n_trees = n_trees
        self.lr = lr
        self.max_depth = max_depth
        self.min_leaf = min_leaf
        self.seed = seed

    def fit(self, X, y, X_val=None, y_val=None):
        self.f0 = float(y.mean())           # the best constant: the minimiser of MSE
        F = np.full(len(y), self.f0)
        self.trees, self.train_loss, self.val_loss = [], [], []
        Fv = np.full(len(X_val), self.f0) if X_val is not None else None
        for _ in range(self.n_trees):
            resid = y - F                   # -dL/dF for L = 0.5 * (y - F)^2
            t = DecisionTree(task='reg', max_depth=self.max_depth,
                             min_leaf=self.min_leaf).fit(X, resid)
            F += self.lr * t.predict(X)
            self.trees.append(t)
            self.train_loss.append(float(np.mean((y - F) ** 2)))
            if Fv is not None:
                Fv += self.lr * t.predict(X_val)
                self.val_loss.append(float(np.mean((y_val - Fv) ** 2)))
        return self

    def predict(self, X, n=None):
        F = np.full(len(X), self.f0)
        for t in self.trees[:n if n is not None else len(self.trees)]:
            F += self.lr * t.predict(X)
        return F


def tabular_reg(n=400, d=6, seed=4, noise=0.6):
    """Three informative columns combined non-linearly, three pure noise columns."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, d))
    y = (np.sin(1.4 * X[:, 0]) + 0.8 * X[:, 1] * X[:, 2] + 0.5 * X[:, 1] ** 2
         + noise * rng.standard_normal(n))
    return X, y


def mse(a, b):
    return float(np.mean((a - b) ** 2))


Xr, yr = tabular_reg(600, 6, seed=4)
rr = np.random.default_rng(2).permutation(600)
rtr, rte = rr[:420], rr[420:]
g = GradientBoosting(n_trees=300, lr=0.1, max_depth=2).fit(Xr[rtr], yr[rtr], Xr[rte], yr[rte])
print('predicting the training mean for everything:',
      round(mse(np.full(len(rte), yr[rtr].mean()), yr[rte]), 4))
print()
print(f'{"trees":>6} {"train MSE":>11} {"test MSE":>10}')
for m in (1, 5, 20, 50, 100, 200, 300):
    print(f'{m:>6} {g.train_loss[m - 1]:>11.4f} {g.val_loss[m - 1]:>10.4f}')

predicting the training mean for everything: 2.6937

 trees   train MSE   test MSE
     1      2.3892     2.5825
     5      2.0205     2.2420
    20      1.1883     1.5372
    50      0.6756     1.0401
   100      0.3816     0.7488
   200      0.2235     0.6325
   300      0.1603     0.6312


## Silent failure #6: the training loss never tells you to stop

Read the `lr = 1.0` row across. Training MSE finishes at visually zero — a perfect fit.
Held-out MSE bottoms out after a couple of dozen trees and then climbs by a third.

The training curve is monotone decreasing for every row in this table — `trees.py`
checks that row by row and prints the result as an extra column. It is structurally
incapable of telling you when to stop, which is why early stopping needs held-out data
and why every boosting library has a parameter for it. None of them require one; they
will all happily run to the end.

Shrinkage is the trade: smaller steps need more trees and reach a better place to stop.
Depth is the other half — a boosting loop wants weak learners, and giving it strong ones
means the first few trees have already memorised the residuals.

In [12]:
print(f'{"lr":>6} {"depth":>6} {"train end":>11} {"best test":>10} {"at tree":>8} {"test end":>9}')
for lr, md_ in [(1.0, 2), (0.3, 2), (0.1, 2), (0.03, 2), (0.1, 6), (0.1, 10)]:
    gb = GradientBoosting(n_trees=300, lr=lr, max_depth=md_).fit(
        Xr[rtr], yr[rtr], Xr[rte], yr[rte])
    b = int(np.argmin(gb.val_loss))
    print(f'{lr:>6} {md_:>6} {gb.train_loss[-1]:>11.4f} {gb.val_loss[b]:>10.4f} '
          f'{b + 1:>8} {gb.val_loss[-1]:>9.4f}')

    lr  depth   train end  best test  at tree  test end
   1.0      2      0.0003     0.8597       20    1.1433
   0.3      2      0.0277     0.6510       95    0.7006
   0.1      2      0.1603     0.6250      250    0.6312
  0.03      2      0.4367     0.8102      300    0.8102
   0.1      6      0.0000     0.6272      142    0.6296
   0.1     10      0.0000     0.7130      132    0.7130


## The forest and the boost pull in opposite directions

Deep and independent kills variance; shallow and dependent kills bias. The forest of
stumps fails because averaging removes variance and a stump has almost none to remove —
it is simply wrong. Boosting on stumps is the best row, because the
dependence is the point: tree 2 is fitted to what tree 1 got wrong. (The two stump rows
are not literally the same trees — the forest gives each stump 3 random columns and the
boosted ones see all 6 — but the depth, which is what the comparison is about, is the
same.)

In [13]:
rows = [
    ('one deep tree (max_depth=None)',
     DecisionTree(task='reg').fit(Xr[rtr], yr[rtr]).predict(Xr[rte])),
    ('one stump (max_depth=2)',
     DecisionTree(task='reg', max_depth=2).fit(Xr[rtr], yr[rtr]).predict(Xr[rte])),
    ('forest of 40 deep trees',
     RandomForest(n_trees=40, task='reg', max_features=3, seed=0)
     .fit(Xr[rtr], yr[rtr]).predict(Xr[rte])),
    ('forest of 40 stumps (max_depth=2)',
     RandomForest(n_trees=40, task='reg', max_depth=2, max_features=3, seed=0)
     .fit(Xr[rtr], yr[rtr]).predict(Xr[rte])),
    ('boosting, 300 stumps, lr 0.1', g.predict(Xr[rte])),
]
print(f'{"model":<42} {"test MSE":>9}')
for nm, pred in rows:
    print(f'{nm:<42} {mse(pred, yr[rte]):>9.4f}')

model                                       test MSE
one deep tree (max_depth=None)                1.1002
one stump (max_depth=2)                       2.0962
forest of 40 deep trees                       0.7165
forest of 40 stumps (max_depth=2)             1.7940
boosting, 300 stumps, lr 0.1                  0.6312


## LeetCode 112 — Path Sum, and LeetCode 257 — Binary Tree Paths

Both problems are about root-to-leaf paths, which is exactly what a prediction and a
model are. Both have a one-character trap.

112: a leaf is a node with **no children**. Recursing into a missing child and testing
the remainder there counts a one-child node as a leaf. A decision tree never hits this
bug, because every internal node it builds gets exactly two children.

257: the accumulator must not be shared. Appending the live list at a leaf stores
something that keeps growing afterwards — Day 39's backtracking bug, in a tree.

In [14]:
# ================================================================ 6. LeetCode
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build(vals):
    """LeetCode's level-order array form, with None for a missing child."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q, i = [root], 1
    while q and i < len(vals):
        node = q.pop(0)
        if i < len(vals):
            if vals[i] is not None:
                node.left = TreeNode(vals[i])
                q.append(node.left)
            i += 1
        if i < len(vals):
            if vals[i] is not None:
                node.right = TreeNode(vals[i])
                q.append(node.right)
            i += 1
    return root


class SolutionPathSum:
    def hasPathSum(self, root, targetSum: int) -> bool:
        # A leaf is a node with NO children.  Recursing into a missing child and
        # testing the remainder there counts a one-child node as a leaf.
        if root is None:
            return False
        if root.left is None and root.right is None:
            return targetSum == root.val
        rest = targetSum - root.val
        return self.hasPathSum(root.left, rest) or self.hasPathSum(root.right, rest)


def has_path_sum_null_leaf(root, target):
    """The spelling that treats "ran out of tree" as "reached a leaf"."""
    if root is None:
        return target == 0
    return (has_path_sum_null_leaf(root.left, target - root.val)
            or has_path_sum_null_leaf(root.right, target - root.val))


class SolutionBinaryTreePaths:
    def binaryTreePaths(self, root) -> list[str]:
        out: list[str] = []

        def walk(node, acc):
            if node is None:
                return
            acc = acc + [str(node.val)]          # a new list, not a mutated one
            if node.left is None and node.right is None:
                out.append('->'.join(acc))
                return
            walk(node.left, acc)
            walk(node.right, acc)

        walk(root, [])
        return out


def binary_tree_paths_shared(root):
    """The same walk with one shared list and no undo - every path comes back as the
    final state of the accumulator instead of its own history (Day 39's bug)."""
    out, acc = [], []

    def walk(node):
        if node is None:
            return
        acc.append(str(node.val))
        if node.left is None and node.right is None:
            out.append(acc)                       # the list itself, not a copy
            return
        walk(node.left)
        walk(node.right)

    walk(root)
    return ['->'.join(p) for p in out]


r22 = build([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1])
print('[5,4,8,11,null,13,4,7,2,null,null,null,1] target 22 ->',
      SolutionPathSum().hasPathSum(r22, 22), ' (the path is 5 -> 4 -> 11 -> 2)')
r12 = build([1, 2])
print('[1,2] target 1  correct               ->', SolutionPathSum().hasPathSum(r12, 1))
print('[1,2] target 1  "None means a leaf"   ->', has_path_sum_null_leaf(r12, 1))
rng12 = np.random.default_rng(11)
bad = 0
for _ in range(3000):
    n = int(rng12.integers(1, 12))
    vals = [int(rng12.integers(-6, 7)) if (i == 0 or rng12.random() < 0.72) else None
            for i in range(n)]
    root = build(vals)
    if root is None:
        continue
    tgt = int(rng12.integers(-12, 13))
    bad += SolutionPathSum().hasPathSum(root, tgt) != has_path_sum_null_leaf(root, tgt)
print(f'over 3,000 random small trees the two disagree {bad} times ({100 * bad / 3000:.1f}%)')
print()
r257 = build([1, 2, 3, None, 5])
print('[1,2,3,null,5] correct       ->', SolutionBinaryTreePaths().binaryTreePaths(r257))
print('[1,2,3,null,5] shared list   ->', binary_tree_paths_shared(r257))

[5,4,8,11,null,13,4,7,2,null,null,null,1] target 22 -> True  (the path is 5 -> 4 -> 11 -> 2)
[1,2] target 1  correct               -> False
[1,2] target 1  "None means a leaf"   -> True
over 3,000 random small trees the two disagree 121 times (4.0%)

[1,2,3,null,5] correct       -> ['1->2->5', '1->3']
[1,2,3,null,5] shared list   -> ['1->2->5->3', '1->2->5->3']


## Tests

Everything above, re-derived from scratch and asserted.

In [15]:
def check() -> None:
    """Everything the walkthrough claims, re-derived from scratch."""
    rng = np.random.default_rng(0)

    # The fast split search agrees with the obvious one, everywhere.
    for _ in range(60):          # 60 problems, each checked under two criteria
        n = int(rng.integers(6, 60))
        d = int(rng.integers(1, 4))
        k = int(rng.integers(2, 4))
        Xr_ = np.round(rng.uniform(0, 3, (n, d)), 2)
        yr_ = rng.integers(0, k, n)
        for crit in ('gini', 'entropy'):
            a_ = best_split_naive(Xr_, yr_, k, crit)
            b_ = best_split(Xr_, yr_, k, crit)
            assert (a_ is None) == (b_ is None)
            if a_ is not None:
                assert abs(a_[0] - b_[0]) < 1e-9
        yv_ = rng.normal(size=n)
        a_ = best_split_naive(Xr_, yv_, None, 'variance')
        b_ = best_split(Xr_, yv_, None, 'variance')
        assert (a_ is None) == (b_ is None)
        if a_ is not None:
            assert abs(a_[0] - b_[0]) < 1e-9

    # Impurity is 0 on a pure node and maximal on a balanced one.
    assert gini(np.zeros(8, int), 2) == 0.0 and entropy(np.zeros(8, int), 2) == 0.0
    assert abs(gini(np.array([0, 0, 0, 0, 1, 1, 1, 1]), 2) - 0.5) < 1e-12
    assert abs(entropy(np.array([0, 0, 0, 0, 1, 1, 1, 1]), 2) - 1.0) < 1e-12

    # A tree grown to purity reproduces its own training labels exactly, on any data
    # set with no contradictory duplicate rows.
    for seed in range(4):
        Xa, ya = stripes(150, seed=seed, noise=0.15)
        ta = DecisionTree().fit(Xa, ya)
        assert accuracy(ta.predict(Xa), ya) == 1.0
    Xs_, ys_ = stripes(400, seed=0, noise=0.10)
    pm = np.random.default_rng(0).permutation(400)
    tr_, te_ = pm[:260], pm[260:]
    accs = [(d, accuracy(DecisionTree(max_depth=d).fit(Xs_[tr_], ys_[tr_]).predict(Xs_[te_]), ys_[te_]))
            for d in (1, 2, 3, 5, 8, 12, None)]
    # training accuracy is monotone in depth; held-out accuracy is not
    tr_accs = [accuracy(DecisionTree(max_depth=d).fit(Xs_[tr_], ys_[tr_]).predict(Xs_[tr_]), ys_[tr_])
               for d in (1, 2, 3, 5, 8, 12, None)]
    assert all(tr_accs[i] <= tr_accs[i + 1] + 1e-12 for i in range(len(tr_accs) - 1))
    assert tr_accs[-1] == 1.0
    assert max(a for _, a in accs) > accs[-1][1]

    # XOR: the correct root split is worth essentially nothing, and greedy misses it.
    Xx_, yx_ = xor_data(400, seed=1)
    par = gini(yx_, 2)
    m = Xx_[:, 0] <= 0.0
    g0 = par - (m.sum() * gini(yx_[m], 2) + (~m).sum() * gini(yx_[~m], 2)) / len(yx_)
    gb_ = best_split(Xx_, yx_, 2, 'gini')[0]
    assert g0 < 1e-4 < gb_
    oracle = ((Xx_[:, 0] > 0) ^ (Xx_[:, 1] > 0)).astype(int)
    assert accuracy(oracle, yx_) == 1.0
    assert accuracy(DecisionTree(max_depth=2).fit(Xx_, yx_).predict(Xx_), yx_) < 0.7

    # Impurity importance prefers the high-cardinality noise column; permutation
    # importance on held-out rows does not.
    Xc_, yc_, _ = cardinality_data(300, seed=2)
    rc_ = np.random.default_rng(0).permutation(300)
    tc_ = DecisionTree().fit(Xc_[rc_[:210]], yc_[rc_[:210]])
    imp_ = tc_.importances(Xc_.shape[1])
    pi_ = permutation_importance(tc_, Xc_[rc_[210:]], yc_[rc_[210:]],
                                 np.random.default_rng(7), repeats=20)
    assert imp_.argmax() == 1 and pi_.argmax() == 0

    # A tree is a step function: it cannot leave the range of its leaf means.
    Xw_, yw_ = wave(120, seed=3)
    tw_ = DecisionTree(task='reg', max_depth=6).fit(Xw_, yw_)
    far = tw_.predict(np.array([[3.5], [5.0], [10.0], [100.0]]))
    assert np.all(far == far[0])
    assert tw_.predict(Xw_).max() < yw_.max()
    fw_ = RandomForest(n_trees=10, task='reg', max_features=None, seed=0).fit(Xw_, yw_)
    far_f = fw_.predict(np.array([[3.5], [5.0], [100.0]]))
    assert np.allclose(far_f, far_f[0])

    # Bagging beats one tree; the forest's trees disagree more than bagging's.
    Xd_, yd_ = dominant(900, seed=0)
    rd_ = np.random.default_rng(1).permutation(900)
    dtr_, dte_ = rd_[:600], rd_[600:]
    one_ = accuracy(DecisionTree().fit(Xd_[dtr_], yd_[dtr_]).predict(Xd_[dte_]), yd_[dte_])
    fb = RandomForest(n_trees=30, seed=0, max_features=None).fit(Xd_[dtr_], yd_[dtr_])
    fs = RandomForest(n_trees=30, seed=0, max_features='sqrt').fit(Xd_[dtr_], yd_[dtr_])
    assert accuracy(fb.predict(Xd_[dte_]), yd_[dte_]) > one_
    assert fs.disagreement(Xd_[dte_]) > fb.disagreement(Xd_[dte_])
    # without feature subsampling every tree opens with the same question
    assert len(set(t.root.feat for t in fb.trees)) == 1
    assert len(set(t.root.feat for t in fs.trees)) > 1
    assert fb.oob_score(Xd_[dtr_], yd_[dtr_])[1] == 600

    # Boosting: the training loss is monotone, the held-out loss is not.
    Xr2, yr2 = tabular_reg(600, 6, seed=4)
    rr_ = np.random.default_rng(2).permutation(600)
    gb2 = GradientBoosting(n_trees=120, lr=1.0, max_depth=2).fit(
        Xr2[rr_[:420]], yr2[rr_[:420]], Xr2[rr_[420:]], yr2[rr_[420:]])
    assert all(gb2.train_loss[i] >= gb2.train_loss[i + 1] - 1e-9
               for i in range(len(gb2.train_loss) - 1))
    b2 = int(np.argmin(gb2.val_loss))
    assert b2 < len(gb2.val_loss) - 1 and gb2.val_loss[-1] > gb2.val_loss[b2] * 1.2
    # the first residual target really is y - mean(y)
    g1 = GradientBoosting(n_trees=1, lr=1.0, max_depth=2).fit(Xr2, yr2)
    assert abs(g1.f0 - yr2.mean()) < 1e-12

    # LeetCode 112 and 257.
    assert SolutionPathSum().hasPathSum(
        build([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, None, 1]), 22) is True
    assert SolutionPathSum().hasPathSum(build([1, 2, 3]), 5) is False
    assert SolutionPathSum().hasPathSum(build([]), 0) is False
    assert SolutionPathSum().hasPathSum(build([1, 2]), 1) is False
    assert has_path_sum_null_leaf(build([1, 2]), 1) is True
    assert sorted(SolutionBinaryTreePaths().binaryTreePaths(build([1, 2, 3, None, 5]))) == \
        ['1->2->5', '1->3']
    assert SolutionBinaryTreePaths().binaryTreePaths(build([1])) == ['1']
    assert binary_tree_paths_shared(build([1, 2, 3, None, 5])) != ['1->2->5', '1->3']


check()
print('all assertions passed')

all assertions passed
